# Base CatBoost Model

In [2]:
# import packages
import seaborn as sns
import torch
import numpy as np
import os
import gc
import pandas as pd
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
from time import time

In [3]:
# Use Cols defined by winner's EDA

# COLUMNS WITH STRINGS
str_type = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain','M1', 'M2', 'M3', 'M4','M5',
            'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 
            'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
str_type += ['id-12', 'id-15', 'id-16', 'id-23', 'id-27', 'id-28', 'id-29', 'id-30', 
            'id-31', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38']

# FIRST 53 COLUMNS
cols = ['TransactionID', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11',
       'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8',
       'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4',
       'M5', 'M6', 'M7', 'M8', 'M9']

# V COLUMNS TO LOAD DECIDED BY CORRELATION EDA
# https://www.kaggle.com/cdeotte/eda-for-columns-v-and-id
v =  [1, 3, 4, 6, 8, 11]
v += [13, 14, 17, 20, 23, 26, 27, 30]
v += [36, 37, 40, 41, 44, 47, 48]
v += [54, 56, 59, 62, 65, 67, 68, 70]
v += [76, 78, 80, 82, 86, 88, 89, 91]

#v += [96, 98, 99, 104] #relates to groups, no NAN 
v += [107, 108, 111, 115, 117, 120, 121, 123] # maybe group, no NAN
v += [124, 127, 129, 130, 136] # relates to groups, no NAN

# LOTS OF NAN BELOW
v += [138, 139, 142, 147, 156, 162] #b1
v += [165, 160, 166] #b1
v += [178, 176, 173, 182] #b2
v += [187, 203, 205, 207, 215] #b2
v += [169, 171, 175, 180, 185, 188, 198, 210, 209] #b2
v += [218, 223, 224, 226, 228, 229, 235] #b3
v += [240, 258, 257, 253, 252, 260, 261] #b3
v += [264, 266, 267, 274, 277] #b3
v += [220, 221, 234, 238, 250, 271] #b3

v += [294, 284, 285, 286, 291, 297] # relates to grous, no NAN
v += [303, 305, 307, 309, 310, 320] # relates to groups, no NAN
v += [281, 283, 289, 296, 301, 314] # relates to groups, no NAN
#v += [332, 325, 335, 338] # b4 lots NAN

cols += ['V'+str(x) for x in v]
dtypes = {}
for c in cols+['id_0'+str(x) for x in range(1,10)]+['id_'+str(x) for x in range(10,34)]+\
    ['id-0'+str(x) for x in range(1,10)]+['id-'+str(x) for x in range(10,34)]:
        dtypes[c] = 'float32'
for c in str_type: dtypes[c] = 'category'

In [4]:
# LOAD TRAIN
X_train = pd.read_csv('../input/ieee-fraud-detection/train_transaction.csv',index_col='TransactionID', dtype=dtypes, usecols=cols+['isFraud'])
train_id = pd.read_csv('../input/ieee-fraud-detection/train_identity.csv',index_col='TransactionID', dtype=dtypes)
X_train = X_train.merge(train_id, how='left', left_index=True, right_index=True)
# LOAD TEST
X_test = pd.read_csv('../input/ieee-fraud-detection/test_transaction.csv',index_col='TransactionID', dtype=dtypes, usecols=cols)
test_id = pd.read_csv('../input/ieee-fraud-detection/test_identity.csv',index_col='TransactionID', dtype=dtypes)
fix = {o:n for o, n in zip(test_id.columns, train_id.columns)}
test_id.rename(columns=fix, inplace=True)
X_test = X_test.merge(test_id, how='left', left_index=True, right_index=True)
# TARGET
y_train = X_train['isFraud'].copy()
del train_id, test_id, X_train['isFraud']; x = gc.collect()
# PRINT STATUS
print('Train shape',X_train.shape,'test shape',X_test.shape)

Train shape (590540, 213) test shape (506691, 213)


In [5]:
# Preprocess data
# NORMALIZE D COLUMNS
for i in range(1,16):
    if i in [1,2,3,5,9]: continue
    X_train['D'+str(i)] =  X_train['D'+str(i)] - X_train.TransactionDT/np.float32(24*60*60)
    X_test['D'+str(i)] = X_test['D'+str(i)] - X_test.TransactionDT/np.float32(24*60*60) 
# Convert M columns to Binary Values
for col in ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']:
    X_train[col] = X_train[col].map({'T': 1, 'F': 0, True: 1, False: 0}).astype('float32')
    X_test[col] = X_test[col].map({'T': 1, 'F': 0, True: 1, False: 0}).astype('float32')

In [6]:
# FREQUENCY ENCODE TOGETHER
def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col],df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df1[nm] = df1[col].map(vc)
        df1[nm] = df1[nm].astype('float32')
        df2[nm] = df2[col].map(vc)
        df2[nm] = df2[nm].astype('float32')
        print(nm,', ',end='')
        
# LABEL ENCODE
def encode_LE(col,train=X_train,test=X_test,verbose=True):
    df_comb = pd.concat([train[col],test[col]],axis=0)
    df_comb,_ = df_comb.factorize(sort=True)
    nm = col
    if df_comb.max()>32000: 
        train[nm] = df_comb[:len(train)].astype('int32')
        test[nm] = df_comb[len(train):].astype('int32')
    else:
        train[nm] = df_comb[:len(train)].astype('int16')
        test[nm] = df_comb[len(train):].astype('int16')
    del df_comb; x=gc.collect()
    if verbose: print(nm,', ',end='')
        
# GROUP AGGREGATION MEAN AND STD
# https://www.kaggle.com/kyakovlev/ieee-fe-with-some-eda
def encode_AG(main_columns, uids, aggregations=['mean'], train_df=X_train, test_df=X_test, 
              fillna=True, usena=False):
    # AGGREGATION OF MAIN WITH UID FOR GIVEN STATISTICS
    for main_column in main_columns:
        print("encoding column: ",main_column)
        for col in uids:
            for agg_type in aggregations:
                new_col_name = main_column+'_'+col+'_'+agg_type
                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col,main_column]]])
                if usena: temp_df.loc[temp_df[main_column]==-1,main_column] = np.nan
                temp_df = temp_df.groupby([col])[main_column].agg([agg_type]).reset_index().rename(
                                                        columns={agg_type: new_col_name})

                temp_df.index = list(temp_df[col])
                temp_df = temp_df[new_col_name].to_dict()   

                train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')
                test_df[new_col_name]  = test_df[col].map(temp_df).astype('float32')
                
                if fillna:
                    # train_df[new_col_name].fillna(-1,inplace=True)
                    # test_df[new_col_name].fillna(-1,inplace=True)
                    train_df[new_col_name] = train_df[new_col_name].fillna(-1)
                    test_df[new_col_name] = test_df[new_col_name].fillna(-1)
                
                print("'"+new_col_name+"'",', ',end='')
                
# COMBINE FEATURES
def encode_CB(col1,col2,df1=X_train,df2=X_test):
    nm = col1+'_'+col2
    df1[nm] = df1[col1].astype(str)+'_'+df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str)+'_'+df2[col2].astype(str) 
    encode_LE(nm,verbose=False)
    print(nm,', ',end='')
    
# GROUP AGGREGATION NUNIQUE
def encode_AG2(main_columns, uids, train_df=X_train, test_df=X_test):
    for main_column in main_columns:  
        for col in uids:
            comb = pd.concat([train_df[[col]+[main_column]],test_df[[col]+[main_column]]],axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col+'_'+main_column+'_ct'] = test_df[col].map(mp).astype('float32')
            print(col+'_'+main_column+'_ct, ',end='')

In [7]:
# TRANSACTION AMT CENTS
X_train['cents'] = (X_train['TransactionAmt'] - np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents'] = (X_test['TransactionAmt'] - np.floor(X_test['TransactionAmt'])).astype('float32')
print('cents, ', end='')
# FREQUENCY ENCODE: ADDR1, CARD1, CARD2, CARD3, P_EMAILDOMAIN
encode_FE(X_train,X_test,['addr1','card1','card2','card3','P_emaildomain'])
# COMBINE COLUMNS CARD1+ADDR1, CARD1+ADDR1+P_EMAILDOMAIN
encode_CB('card1','addr1')
encode_CB('card1_addr1','P_emaildomain')
# FREQUENCY ENOCDE
encode_FE(X_train,X_test,['card1_addr1','card1_addr1_P_emaildomain'])
# GROUP AGGREGATE
encode_AG(['TransactionAmt','D9','D11'],['card1','card1_addr1','card1_addr1_P_emaildomain'],['mean','std'],usena=True)

import datetime
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
X_train['DT_M'] = X_train['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_train['DT_M'] = (X_train['DT_M'].dt.year-2017)*12 + X_train['DT_M'].dt.month 

X_test['DT_M'] = X_test['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_test['DT_M'] = (X_test['DT_M'].dt.year-2017)*12 + X_test['DT_M'].dt.month

cents, addr1_FE , card1_FE , card2_FE , card3_FE , P_emaildomain_FE , card1_addr1 , card1_addr1_P_emaildomain , card1_addr1_FE , card1_addr1_P_emaildomain_FE , encoding column:  TransactionAmt
'TransactionAmt_card1_mean' , 'TransactionAmt_card1_std' , 'TransactionAmt_card1_addr1_mean' , 'TransactionAmt_card1_addr1_std' , 'TransactionAmt_card1_addr1_P_emaildomain_mean' , 'TransactionAmt_card1_addr1_P_emaildomain_std' , encoding column:  D9
'D9_card1_mean' , 'D9_card1_std' , 'D9_card1_addr1_mean' , 'D9_card1_addr1_std' , 'D9_card1_addr1_P_emaildomain_mean' , 'D9_card1_addr1_P_emaildomain_std' , encoding column:  D11
'D11_card1_mean' , 'D11_card1_std' , 'D11_card1_addr1_mean' , 'D11_card1_addr1_std' , 'D11_card1_addr1_P_emaildomain_mean' , 'D11_card1_addr1_P_emaildomain_std' , 

In [8]:
# Add UID
X_train['day'] = X_train.TransactionDT / (24*60*60)
X_train['uid'] = X_train.card1_addr1.astype(str)+'_'+np.floor(X_train.day-X_train.D1).astype(str)

X_test['day'] = X_test.TransactionDT / (24*60*60)
X_test['uid'] = X_test.card1_addr1.astype(str)+'_'+np.floor(X_test.day-X_test.D1).astype(str)

In [9]:
# FREQUENCY ENCODE UID
encode_FE(X_train,X_test,['uid'])
# AGGREGATE 
encode_AG(['TransactionAmt','D4','D9','D10','D15'],['uid'],['mean','std'],fillna=True,usena=True)
# AGGREGATE
encode_AG(['C'+str(x) for x in range(1,15) if x!=3],['uid'],['mean'],X_train,X_test,fillna=True,usena=True)
# AGGREGATE
encode_AG(['M'+str(x) for x in range(1,10)],['uid'],['mean'],fillna=True,usena=True)
# AGGREGATE
encode_AG2(['P_emaildomain','dist1','DT_M','id_02','cents'], ['uid'], train_df=X_train, test_df=X_test)
# AGGREGATE
encode_AG(['C14'],['uid'],['std'],X_train,X_test,fillna=True,usena=True)
# AGGREGATE 
encode_AG2(['C13','V314'], ['uid'], train_df=X_train, test_df=X_test)
# AGGREATE 
encode_AG2(['V127','V136','V309','V307','V320'], ['uid'], train_df=X_train, test_df=X_test)
# NEW FEATURE
X_train['outsider15'] = (np.abs(X_train.D1-X_train.D15)>3).astype('int8')
X_test['outsider15'] = (np.abs(X_test.D1-X_test.D15)>3).astype('int8')

uid_FE , encoding column:  TransactionAmt
'TransactionAmt_uid_mean' , 'TransactionAmt_uid_std' , encoding column:  D4
'D4_uid_mean' , 'D4_uid_std' , encoding column:  D9
'D9_uid_mean' , 'D9_uid_std' , encoding column:  D10
'D10_uid_mean' , 'D10_uid_std' , encoding column:  D15
'D15_uid_mean' , 'D15_uid_std' , encoding column:  C1
'C1_uid_mean' , encoding column:  C2
'C2_uid_mean' , encoding column:  C4
'C4_uid_mean' , encoding column:  C5
'C5_uid_mean' , encoding column:  C6
'C6_uid_mean' , encoding column:  C7
'C7_uid_mean' , encoding column:  C8
'C8_uid_mean' , encoding column:  C9
'C9_uid_mean' , encoding column:  C10
'C10_uid_mean' , encoding column:  C11
'C11_uid_mean' , encoding column:  C12
'C12_uid_mean' , encoding column:  C13
'C13_uid_mean' , encoding column:  C14
'C14_uid_mean' , encoding column:  M1
'M1_uid_mean' , encoding column:  M2
'M2_uid_mean' , encoding column:  M3
'M3_uid_mean' , encoding column:  M4
'M4_uid_mean' , encoding column:  M5
'M5_uid_mean' , encoding colu

/tmp/ipykernel_55/3128549550.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')


uid_cents_ct, encoding column:  C14


/tmp/ipykernel_55/3128549550.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')


'C14_uid_std' , 

...

/tmp/ipykernel_55/3128549550.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')


uid_V307_ct, 

/tmp/ipykernel_55/3128549550.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')


uid_V320_ct, 

/tmp/ipykernel_55/3128549550.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col+'_'+main_column+'_ct'] = test_df[col].map(mp).astype('float32')
/tmp/ipykernel_55/2971023361.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train['outsider15'] = (np.abs(X_train.D1-X_train.D15)>3).astype('int8')
/tmp/ipykernel_55/2971023361.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once 

In [10]:
cols = list( X_train.columns )
cols.remove('TransactionDT')
for c in ['D6','D7','D8','D9','D12','D13','D14']:
    cols.remove(c)
for c in ['DT_M','day','uid']:
    cols.remove(c)
    
# FAILED TIME CONSISTENCY TEST
for c in ['C3','M5','id_08','id_33']:
    cols.remove(c)
for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    cols.remove(c)
for c in ['id_'+str(x) for x in range(22,28)]:
    cols.remove(c)

print('USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

USING THE FOLLOWING 263 FEATURES.


array(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5',
       'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain',
       'R_emaildomain', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
       'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5',
       'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4', 'M6', 'M7', 'M8',
       'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17',
       'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41',
       'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67',
       'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89',
       'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121',
       'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139',
       'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169',
       'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185',
       'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210',
       '

In [11]:
import joblib
import json
import os

# Create directory for model artifacts
model_dir = '/kaggle/working/catboost'
os.makedirs(model_dir, exist_ok=True)

In [12]:
oof = np.zeros(len(X_train))
preds = np.zeros(len(X_test))
fold_models = []  # Keep in memory for threshold tuning

In [13]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

def evaluate_fraud_model(y_true, probabilities, amounts, thresholds=None):
    """
    Evaluate fraud model with both count-based and dollar-weighted metrics.
    
    Parameters:
    - y_true: actual labels
    - probabilities: predicted probabilities
    - amounts: transaction amounts
    - thresholds: list of thresholds to evaluate (default: 0.1 to 0.9)
    """
    if thresholds is None:
        thresholds = np.arange(0.1, 0.95, 0.05)
    
    results = []
    
    total_fraud_count = y_true.sum()
    total_fraud_dollars = amounts[y_true == 1].sum()
    total_legit_count = (y_true == 0).sum()
    total_legit_dollars = amounts[y_true == 0].sum()
    
    for thresh in thresholds:
        preds = (probabilities >= thresh).astype(int)
        
        # Basic confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        
        # Count-based metrics
        fnr = fn / (fn + tp)  # false negative rate
        fpr = fp / (fp + tn)  # false positive rate
        recall = tp / (tp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Dollar-weighted metrics
        fraud_mask = y_true == 1
        legit_mask = y_true == 0
        
        missed_fraud_dollars = amounts[(fraud_mask) & (preds == 0)].sum()
        caught_fraud_dollars = amounts[(fraud_mask) & (preds == 1)].sum()
        blocked_legit_dollars = amounts[(legit_mask) & (preds == 1)].sum()
        
        dollar_weighted_fnr = missed_fraud_dollars / total_fraud_dollars
        dollar_weighted_recall = caught_fraud_dollars / total_fraud_dollars
        dollar_weighted_fpr = blocked_legit_dollars / total_legit_dollars
        
        results.append({
            'threshold': thresh,
            'recall': recall,
            'fnr': fnr,
            'precision': precision,
            'fpr': fpr,
            'missed_fraud_count': fn,
            'missed_fraud_dollars': missed_fraud_dollars,
            'dollar_weighted_recall': dollar_weighted_recall,
            'dollar_weighted_fnr': dollar_weighted_fnr,
            'blocked_legit_dollars': blocked_legit_dollars,
            'flagged_transactions': fp + tp
        })
    
    return pd.DataFrame(results)

def find_optimal_threshold(y_true, probabilities, amounts, min_recall=0.95, use_dollar_weighted=True):
    """
    Find threshold that achieves minimum recall while minimizing false positives.
    """
    results = evaluate_fraud_model(y_true, probabilities, amounts)
    
    recall_col = 'dollar_weighted_recall' if use_dollar_weighted else 'recall'
    
    # Filter to thresholds meeting recall constraint
    valid = results[results[recall_col] >= min_recall]
    
    if len(valid) == 0:
        print(f"Warning: No threshold achieves {min_recall:.0%} recall. Returning lowest threshold.")
        return results.iloc[0]['threshold'], results
    
    # Among valid thresholds, pick the one with lowest FPR (highest threshold)
    optimal = valid.loc[valid['fpr'].idxmin()]
    
    return optimal['threshold'], results

# Usage with your OOF predictions
results_df = evaluate_fraud_model(
    y_true=y_train,
    probabilities=oof,
    amounts=X_train['TransactionAmt']
)

In [18]:
skf = GroupKFold(n_splits=6)

cat_features = [col for col in cols if X_train[col].dtype == 'object' or X_train[col].dtype.name == 'category']

for col in cat_features:
    # Convert to string first, then fill NaN
    X_train[col] = X_train[col].astype(str).replace('nan', 'missing')
    X_test[col] = X_test[col].astype(str).replace('nan', 'missing')

for i, (idxT, idxV) in enumerate(skf.split(X_train, y_train, groups=X_train['DT_M'])):
    month = X_train.iloc[idxV]['DT_M'].iloc[0]
    print('Fold', i, 'withholding month', month)
    print(' rows of train =', len(idxT), 'rows of holdout =', len(idxV))
    
    # Identify categorical columns if any
    cat_features = [col for col in cols if X_train[col].dtype == 'object' or X_train[col].dtype.name == 'category']
    
    clf = CatBoostClassifier(
        iterations=5000,
        depth=12,
        learning_rate=0.02,
        # subsample=0.8,
        bootstrap_type='Bernoulli',
        eval_metric='AUC',
        cat_features=cat_features if cat_features else None,
        early_stopping_rounds=100,
        task_type='GPU',
        devices='0',
        verbose=100,
        random_seed=42,
        # For imbalanced data - choose ONE of these:
        auto_class_weights='Balanced'
    )
    
    clf.fit(
        X_train[cols].iloc[idxT], y_train.iloc[idxT],
        eval_set=(X_train[cols].iloc[idxV], y_train.iloc[idxV]),
    )
    
    oof[idxV] += clf.predict_proba(X_train[cols].iloc[idxV])[:, 1]
    preds += clf.predict_proba(X_test[cols])[:, 1] / skf.n_splits
    
    # Save each fold model
    model_path = f'{model_dir}/catboost_fold_{i}.cbm'
    clf.save_model(model_path)  # CatBoost native format
    print(f' Saved model to {model_path}')
    
    fold_models.append(clf)
    _ = gc.collect()

print('#' * 20)
print('CatBoost OOF CV=', roc_auc_score(y_train, oof))

# Find optimal threshold using your evaluation function
optimal_thresh, results_df = find_optimal_threshold(
    y_true=y_train,
    probabilities=oof,
    amounts=X_train['TransactionAmt'],
    min_recall=0.95,
    use_dollar_weighted=True
)
print(f'Optimal threshold: {optimal_thresh}')

# Save metadata needed for inference
metadata = {
    'feature_columns': list(cols),
    'cat_features': cat_features,
    'n_folds': skf.n_splits,
    'threshold': float(optimal_thresh),
    'oof_auc': float(roc_auc_score(y_train, oof)),
    'model_params': {
        'iterations': 5000,
        'depth': 12,
        'learning_rate': 0.02,
        'subsample': 0.8,
        'rsm': 0.4
    }
}
with open(f'{model_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Save OOF predictions for later analysis
np.save(f'{model_dir}/oof_predictions_catboost.npy', oof)
np.save(f'{model_dir}/test_predictions_catboost.npy', preds)
np.save(f'{model_dir}/y_train_catboost.npy', y_train.values)
np.save(f'{model_dir}/transaction_amounts_catboost.npy', X_train['TransactionAmt'].values)
np.save(f'{model_dir}/months_catboost.npy', X_train['DT_M'].values)

Fold 0 withholding month 12
 rows of train = 453219 rows of holdout = 137321


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8199405	best: 0.8199405 (0)	total: 678ms	remaining: 56m 28s
100:	test: 0.9029850	best: 0.9029850 (100)	total: 42.4s	remaining: 34m 15s
200:	test: 0.9132834	best: 0.9132834 (200)	total: 1m 24s	remaining: 33m 28s
300:	test: 0.9168755	best: 0.9170154 (298)	total: 2m 5s	remaining: 32m 43s
400:	test: 0.9192991	best: 0.9195872 (385)	total: 2m 47s	remaining: 32m 3s
500:	test: 0.9196494	best: 0.9203250 (471)	total: 3m 29s	remaining: 31m 22s
bestTest = 0.9203250408
bestIteration = 471
Shrink model to first 472 iterations.
 Saved model to /kaggle/working/catboost/catboost_fold_0.cbm
Fold 1 withholding month 15
 rows of train = 488908 rows of holdout = 101632


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8460682	best: 0.8460682 (0)	total: 533ms	remaining: 44m 23s
100:	test: 0.9358870	best: 0.9358870 (100)	total: 45.4s	remaining: 36m 41s
200:	test: 0.9457479	best: 0.9457479 (200)	total: 1m 29s	remaining: 35m 41s
300:	test: 0.9488042	best: 0.9488139 (297)	total: 2m 13s	remaining: 34m 48s
400:	test: 0.9497256	best: 0.9497256 (400)	total: 2m 58s	remaining: 34m 4s
500:	test: 0.9499807	best: 0.9500166 (498)	total: 3m 42s	remaining: 33m 17s
600:	test: 0.9499644	best: 0.9502270 (519)	total: 4m 26s	remaining: 32m 29s
bestTest = 0.9502270222
bestIteration = 519
Shrink model to first 520 iterations.
 Saved model to /kaggle/working/catboost/catboost_fold_1.cbm
Fold 2 withholding month 13
 rows of train = 497955 rows of holdout = 92585


...

0:	test: 0.8355648	best: 0.8355648 (0)	total: 509ms	remaining: 42m 26s
100:	test: 0.9246757	best: 0.9246757 (100)	total: 45.9s	remaining: 37m 4s
200:	test: 0.9363963	best: 0.9363963 (200)	total: 1m 30s	remaining: 36m 7s
300:	test: 0.9393901	best: 0.9393901 (300)	total: 2m 15s	remaining: 35m 14s
400:	test: 0.9404049	best: 0.9404049 (400)	total: 3m	remaining: 34m 29s
500:	test: 0.9406111	best: 0.9406834 (465)	total: 3m 45s	remaining: 33m 42s
600:	test: 0.9405695	best: 0.9408915 (525)	total: 4m 29s	remaining: 32m 55s
bestTest = 0.9408915043
bestIteration = 525
Shrink model to first 526 iterations.
 Saved model to /kaggle/working/catboost/catboost_fold_3.cbm
Fold 4 withholding month 14
 rows of train = 504519 rows of holdout = 86021


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8700106	best: 0.8700106 (0)	total: 504ms	remaining: 41m 59s
100:	test: 0.9426150	best: 0.9426150 (100)	total: 46.2s	remaining: 37m 20s
200:	test: 0.9540921	best: 0.9540921 (200)	total: 1m 31s	remaining: 36m 25s
300:	test: 0.9559885	best: 0.9561737 (273)	total: 2m 16s	remaining: 35m 28s
400:	test: 0.9570557	best: 0.9570557 (400)	total: 3m 1s	remaining: 34m 39s
500:	test: 0.9573902	best: 0.9574744 (496)	total: 3m 46s	remaining: 33m 51s
600:	test: 0.9572505	best: 0.9577356 (547)	total: 4m 31s	remaining: 33m 4s
bestTest = 0.9577355981
bestIteration = 547
Shrink model to first 548 iterations.
 Saved model to /kaggle/working/catboost/catboost_fold_4.cbm
Fold 5 withholding month 16
 rows of train = 506885 rows of holdout = 83655


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8383340	best: 0.8383340 (0)	total: 507ms	remaining: 42m 12s
100:	test: 0.9419881	best: 0.9419881 (100)	total: 46.4s	remaining: 37m 29s
200:	test: 0.9578695	best: 0.9578695 (200)	total: 1m 31s	remaining: 36m 32s
300:	test: 0.9612660	best: 0.9612660 (300)	total: 2m 17s	remaining: 35m 43s
400:	test: 0.9617478	best: 0.9618637 (335)	total: 3m 2s	remaining: 34m 57s
500:	test: 0.9618691	best: 0.9621304 (448)	total: 3m 48s	remaining: 34m 8s
bestTest = 0.9621304274
bestIteration = 448
Shrink model to first 449 iterations.
 Saved model to /kaggle/working/catboost/catboost_fold_5.cbm
####################
CatBoost OOF CV= 0.9459378218907504
Optimal threshold: 0.1


In [19]:
import shutil
shutil.make_archive('/kaggle/working/catboost', 'zip', '/kaggle/working/catboost')

'/kaggle/working/catboost.zip'